# Fallen Tree Detection -- YOLO26n Training, Validation & Testing

**Phase 1 scope only** -- this notebook trains and evaluates a single-class
"fallen_tree" detector. It does NOT build any backend/frontend/API.

**Dataset source:** [Roboflow Universe -- fallen-tree-e4f2f](https://universe.roboflow.com/hazard-detection-kzrtt/fallen-tree-e4f2f)
(workspace `hazard-detection-kzrtt`, project `fallen-tree-e4f2f`)

**Known public metadata (verify against the actual downloaded `data.yaml` in Step 3 -- do not trust this blindly):**
- Only **63 images total** (version 3) -- this is a SMALL dataset for object detection
- Split: ~40 train / 14 valid / 9 test
- 1 class observed on the project page: `fallen_tree`
- License CC BY 4.0; preprocessing applied by the dataset author: auto-orient + resize (stretch) to 640x640
- Multiple dataset versions exist on Roboflow; this notebook lists them and lets you pick

**IMPORTANT CAVEAT:** With only ~60 images, metrics from this model should be
treated as a weak prototype signal, not a production-quality result. This
notebook relies on transfer learning from COCO-pretrained YOLO26n weights and
Ultralytics' default augmentations (mosaic, flips, HSV jitter, etc.) to make
the most of limited data, but do not expect production-grade recall/precision.
If Phase 1 evaluation shows this model is too weak to be useful, the
documented next step is to source additional fallen-tree images before
Phase 2, not to fabricate better numbers.

**Run this in Google Colab with a GPU runtime**: `Runtime > Change runtime type > T4 GPU`.

**Credential setup (do this before running, do NOT paste your key into any cell):**
1. Get a free Roboflow API key: https://app.roboflow.com/settings/api
2. In this Colab notebook, open the key icon in the left sidebar ("Secrets")
3. Add a secret named exactly `ROBOFLOW_API_KEY` with your key as the value, and toggle "Notebook access" on
4. Do not commit this key anywhere in the repository


## Step 1 -- Install dependencies

In [ ]:
!pip install -q "ultralytics>=8.3" roboflow
import ultralytics
ultralytics.checks()


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. In Colab: Runtime > Change runtime type > T4 GPU, then re-run this cell.")


## Step 2 -- Download the dataset via the Roboflow API (authenticated, not scraped)

In [ ]:
from roboflow import Roboflow
from google.colab import userdata

api_key = userdata.get("ROBOFLOW_API_KEY")
assert api_key, "Set the ROBOFLOW_API_KEY Colab secret first (see markdown above)."

rf = Roboflow(api_key=api_key)
project = rf.workspace("hazard-detection-kzrtt").project("fallen-tree-e4f2f")

available_versions = [v.version for v in project.versions()]
print("Available versions:", available_versions)

VERSION = max(available_versions) if available_versions else 3
dataset = project.version(VERSION).download("yolov8")
print("Downloaded version", VERSION, "to:", dataset.location)


## Step 3 -- Inspect the dataset (ground truth, not the webpage)

In [ ]:
import yaml, os, glob, json

DATASET_DIR = dataset.location
with open(os.path.join(DATASET_DIR, "data.yaml")) as f:
    data_cfg = yaml.safe_load(f)
print(json.dumps(data_cfg, indent=2))

total = 0
for split in ("train", "val", "test"):
    key = data_cfg.get(split)
    if not key:
        print(f"{split}: NOT DEFINED in data.yaml")
        continue
    img_dir = key if os.path.isabs(key) else os.path.join(DATASET_DIR, key)
    n = len(glob.glob(os.path.join(img_dir, "*.*")))
    total += n
    print(f"{split}: {n} images ({img_dir})")
print(f"TOTAL: {total} images -- if this is under ~100, treat results as a weak prototype signal only.")


## Step 4 -- Verify labels and class names (mandatory -- do not skip)

We do NOT assume the dataset's class names already match our canonical
hazard label `fallen_tree`.


In [ ]:
from collections import Counter

label_files = glob.glob(os.path.join(DATASET_DIR, "**", "labels", "*.txt"), recursive=True)
print(f"Found {len(label_files)} label files")

class_id_counts = Counter()
for lf in label_files:
    with open(lf) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            class_id_counts[int(line.split()[0])] += 1

names = data_cfg["names"]
print("Classes declared in data.yaml:", names)
for cid, cnt in sorted(class_id_counts.items()):
    label = names[cid] if cid < len(names) else "<UNKNOWN INDEX>"
    print(f"  id={cid} name={label!r} instances={cnt}")

unexpected = [cid for cid in class_id_counts if cid >= len(names)]
assert not unexpected, f"Label files reference class ids not in data.yaml: {unexpected}. STOP and investigate manually."


## Step 5 -- Class normalization

Canonical target: exactly one class, `fallen_tree` (index 0). If the
dataset ships extra/different class names, print them and stop for manual
review rather than silently discarding.


In [ ]:
CANONICAL_NAMES = ["fallen_tree"]

declared_names = data_cfg["names"]
print("Declared names:", declared_names)

if declared_names == CANONICAL_NAMES:
    print("Dataset already matches the canonical single-class schema. No remapping needed.")
else:
    print("MISMATCH -- manual review required. Declared names differ from canonical schema.")
    print("Declared:", declared_names)
    print("Canonical:", CANONICAL_NAMES)
    raise SystemExit("Resolve the class mismatch above before proceeding to training.")


## Step 6 -- Prepare the YOLO data config

In [ ]:
DATA_YAML = os.path.join(DATASET_DIR, "data.yaml")

for split in ("train", "val", "test"):
    if split in data_cfg and data_cfg[split] and not os.path.isabs(data_cfg[split]):
        data_cfg[split] = os.path.join(DATASET_DIR, data_cfg[split])

data_cfg["nc"] = 1
data_cfg["names"] = CANONICAL_NAMES

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(data_cfg, f)

print(open(DATA_YAML).read())


## Step 7 -- Load the model

In [ ]:
from ultralytics import YOLO

# YOLO26n (nano) is the current lightweight Ultralytics model (released Jan 2026,
# ~2.4M params, NMS-free end-to-end head) -- chosen for fast prototype training and
# fast CPU/edge inference later. Falls back to YOLO11n if the installed
# ultralytics version does not yet ship YOLO26 weights.
MODEL_WEIGHTS = "yolo26n.pt"
try:
    model = YOLO(MODEL_WEIGHTS)
except Exception as e:
    print(f"Could not load {MODEL_WEIGHTS} ({e}); falling back to yolo11n.pt")
    MODEL_WEIGHTS = "yolo11n.pt"
    model = YOLO(MODEL_WEIGHTS)
print("Loaded:", MODEL_WEIGHTS)


## Step 8 -- Train

Given the small dataset (~60 images), consider a higher epoch count with
early-stopping patience so the model can extract as much signal as possible
without you manually babysitting it; Ultralytics' default augmentation
pipeline (mosaic, flips, HSV jitter) is left enabled to mitigate the small
sample size.


In [ ]:
EPOCHS = 150
IMGSZ = 640
BATCH = 16
PATIENCE = 30

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    project="runs/fallen_tree",
    name="train",
    exist_ok=True,
    plots=True,
)


## Step 9 -- Validate (on the VAL split, separate from training)

In [ ]:
val_metrics = model.val(data=DATA_YAML, split="val")

print("=== Validation metrics ===")
print("precision(B):", val_metrics.box.mp)
print("recall(B):   ", val_metrics.box.mr)
print("mAP50:       ", val_metrics.box.map50)
print("mAP50-95:    ", val_metrics.box.map)

# Ultralytics writes confusion_matrix.png, PR_curve.png, etc. into the val run dir.
import glob
val_dir = val_metrics.save_dir
print("Validation artifacts saved to:", val_dir)
for p in sorted(glob.glob(str(val_dir) + "/*.png")):
    print(" -", p)


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

cm_path = str(val_metrics.save_dir / "confusion_matrix.png")
try:
    img = Image.open(cm_path)
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Confusion matrix (validation set)")
    plt.show()
except FileNotFoundError:
    print("confusion_matrix.png not found -- with a single class it may not be generated by all Ultralytics versions.")


## Step 10 -- Test on unseen images (TEST split, never used for train or val)

In [ ]:
# Final held-out TEST evaluation -- these images were never used for training or
# validation. This is the mandatory "unseen data" evaluation for Phase 1.
test_metrics = model.val(data=DATA_YAML, split="test")

print("=== TEST metrics (unseen images) ===")
print("precision(B):", test_metrics.box.mp)
print("recall(B):   ", test_metrics.box.mr)
print("mAP50:       ", test_metrics.box.map50)
print("mAP50-95:    ", test_metrics.box.map)
print("Artifacts:   ", test_metrics.save_dir)


In [ ]:
import glob, random
from ultralytics import YOLO

test_images = sorted(glob.glob("os.path.join(DATASET_DIR, data_cfg['test'], '*.*')"))
print(f"Found {len(test_images)} unseen test images")

sample = random.sample(test_images, min(8, len(test_images)))
pred_results = model.predict(source=sample, conf=0.25, save=True, project="runs/predict", name="samples", exist_ok=True)

import matplotlib.pyplot as plt
from PIL import Image

n = len(pred_results)
cols = min(4, n) if n else 1
rows = (n + cols - 1) // cols if n else 1
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
axes = axes.flatten() if n > 1 else [axes]
for ax, res in zip(axes, pred_results):
    im = Image.fromarray(res.plot()[..., ::-1])
    ax.imshow(im)
    ax.axis("off")
    n_det = len(res.boxes)
    ax.set_title(f"{n_det} detection(s)")
for ax in axes[len(pred_results):]:
    ax.axis("off")
plt.suptitle("Sample predictions on unseen TEST images -- hazard: fallen_tree")
plt.tight_layout()
plt.show()


## Step 11 -- Metrics summary (real numbers only -- do not hand-edit)

In [ ]:
import json

summary = {
    "hazard": "fallen_tree",
    "model_weights": MODEL_WEIGHTS,
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "val_precision": float(val_metrics.box.mp),
    "val_recall": float(val_metrics.box.mr),
    "val_map50": float(val_metrics.box.map50),
    "val_map50_95": float(val_metrics.box.map),
    "test_precision": float(test_metrics.box.mp),
    "test_recall": float(test_metrics.box.mr),
    "test_map50": float(test_metrics.box.map50),
    "test_map50_95": float(test_metrics.box.map),
}
print(json.dumps(summary, indent=2))

with open("training_summary_fallen_tree.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Saved training_summary_fallen_tree.json -- copy this into docs/PHASE1.md's results table as real evidence.")


## Step 12 -- Export best.pt

In [ ]:
import shutil, os

HAZARD = "fallen_tree"
best_pt_src = str(model.trainer.best) if hasattr(model, "trainer") and model.trainer is not None else "runs/fallen_tree/train/weights/best.pt"
print("Best weights produced at:", best_pt_src)

# Intended final repo-relative location (NOT committed to git -- see docs/PHASE1.md):
#   ai/models/fallen_tree/best.pt
local_target_dir = f"ai_models_{HAZARD}"
os.makedirs(local_target_dir, exist_ok=True)
local_target = os.path.join(local_target_dir, "best.pt")
shutil.copy(best_pt_src, local_target)
print("Copied to:", local_target)

# Optional: persist to Google Drive so the weight survives the Colab session.
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    drive_dir = f"/content/drive/MyDrive/ResQDrive/models/{HAZARD}"
    os.makedirs(drive_dir, exist_ok=True)
    shutil.copy(best_pt_src, os.path.join(drive_dir, "best.pt"))
    print("Also copied to Google Drive:", drive_dir)
except Exception as e:
    print("Drive not mounted / not in Colab -- skipping Drive copy:", e)

# Download to your machine (place it at ai/models/fallen_tree/best.pt in the repo).
try:
    from google.colab import files
    files.download(local_target)
except Exception as e:
    print("files.download unavailable in this environment:", e)


## Done

Copy `training_summary_fallen_tree.json`'s numbers into `docs/PHASE1.md`'s
results table, and place the downloaded `best.pt` at
`ai/models/fallen_tree/best.pt` in the repo (git-ignored by design).

**Reminder:** with only ~60 source images, if TEST mAP50 is very low (e.g.
below ~0.3) or predictions look unreliable on the sample grid above, that is
a legitimate "blocking dataset issue" per Phase 1's rules -- document it in
docs/PHASE1.md rather than treating the model as production-ready.
